# Importing Libraries

In [ ]:
import pandas as pd

# Loading Env Files

In [ ]:
# 1. Load the .env file
load_dotenv()

# Reading the csv file

In [ ]:

emails = pd.read_csv('emails.csv')
emails.head()

/tmp/ipykernel_24932/2408689559.py:3: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  emails = pd.read_csv('emails.csv')


,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


# Checking the Numeric Columns and converting it into the Type Number

In [537]:

numeric_cols = ['crm_contractor_sentiment_score', 'crm_agent_chase_count']
for col in numeric_cols:
    emails[col] = pd.to_numeric(emails[col], errors='coerce')

emails[numeric_cols].head()

,crm_contractor_sentiment_score,crm_agent_chase_count
0,50.0,1.0
1,NaN,1.0
2,50.0,0.0
3,80.0,3.0
4,80.0,0.0


# Handling Numeric Columns

In [538]:
for col in numeric_cols:
    emails[col] = emails[col].fillna(0)

emails[numeric_cols].isnull().sum()

,0
crm_contractor_sentiment_score,0
crm_agent_chase_count,0


# Handle categorical columns

In [539]:

categorical_cols = emails.columns.difference(numeric_cols)

for col in categorical_cols[1:]:
    emails[col] = emails[col].astype(str).str.strip().str.lower()
    emails[col] = emails[col].replace(['nan', '', 'none', np.nan], 'not discussed')

emails[categorical_cols].head()

,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_accreditation_issues,crm_agent_chased_contractor,crm_auto_renewal_status,crm_competitors_mentioned,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_suggested_leave,...,crm_dts_or_ssip_mentioned,crm_financial_hardship_mentioned,crm_membership_level,crm_membership_overdue,crm_negative_customer_experience,crm_platform_issues_raised,crm_progress_towards_accreditation,crm_refund_mentioned,crm_timely_completion,year
0,KG5766,pre_renewal,not discussed,not discussed,yes,0,no,yes,neutral,no,...,no,yes,not discussed,yes,yes,no,not discussed,yes,not discussed,2025
1,EJ1532,14_out,not discussed,not discussed,yes,0,no,no,not discussed,not discussed,...,no,not discussed,accredited,not discussed,yes,no,not discussed,yes,not discussed,2025
2,AA4063,prior_year,not discussed,not discussed,no,0,no,yes,neutral,no,...,no,not discussed,accredited,no,yes,no,not discussed,yes,not discussed,2025
3,JY9888,prior_year,no,not discussed,yes,0,no,yes,satisfied,no,...,yes,not discussed,accredited,yes,yes,yes,not discussed,yes,no,2025
4,WO6689,pre_renewal,not discussed,no,no,0,no,yes,satisfied,no,...,no,not discussed,accredited,no,yes,no,not discussed,yes,not discussed,2026


# Handle crm_delays_in_accreditation Column

In [540]:

def clean_delay_value(x):
    if pd.isna(x):
        return "not discussed"
    x_low = x.strip().lower()

    if "not discussed" in x_low:
        return "not discussed"

    # Rule 2: contains yes
    if "yes" in x_low:
        return "yes"

    # Rule 3: contains no
    if "no" in x_low:
        return "no"
    return "not discussed"

# Apply cleaning
df["crm_delays_in_accreditation_clean"] = emails["crm_delays_in_accreditation"].apply(clean_delay_value)
# Correcting the typo from 'crm_delays_in_accreditaion' to 'crm_delays_in_accreditation'
# This updates the original column with the cleaned values.
emails["crm_delays_in_accreditation"] = df["crm_delays_in_accreditation_clean"]


# Handle crm_contractor_engagement Column

In [541]:

def clean_engagement_value(text):
    if pd.isna(text) or text is None:
       return 'Not Discussed'
    text_low = str(text).strip().lower()
    no_keywords = [
        'price is too high',
        'dissatisfaction',
        'cost',
        'not feasible to renew',
        'not wish to renew',
        'drop clients',
        'time constraints',
        'difficulties in responding',
        'frustration',
        'cancel',
        'ridiculous',
        'unhappy with no-refund',
        'threatening to look elsewhere',
        'loss of interest',
        'change in business structure',
        'irrelevant requests',
        'not the right time'
    ]

    for keyword in no_keywords:
        if keyword in text_low:
            return 'no'
    not_discussed_keywords = [
        'not discussed',
        'just asked for invoice',
        'discussion about a one-off saving'
    ]

    for keyword in not_discussed_keywords:
        if keyword in text_low:
            return 'not discussed'
    return 'yes'

df['crm_contractor_engagement_clean'] = emails['crm_contractor_engagement'].apply(clean_engagement_value)

emails["crm_contractor_engagement"] = df["crm_contractor_engagement_clean"]


# Handle crm_contractor_sentiment Column Values

In [542]:
def clean_contractor_sentiment(text):
    # k2YDnG0gUtwl already converts to string, lowercases, and handles NaNs by replacing with 'not discussed'.
    # So, we can directly work with the lowercased string.
    text_low = str(text).strip().lower()

    # Stronger negative sentiment keywords (these should take precedence)
    dissatisfied_keywords = [
        'dissatisfied', 'cancellation', 'ridiculous', 'no work gained',
        'unhappy', 'price is too high', 'frustration', 'difficulties', 'cost',
        'not feasible to renew', 'not wish to renew', 'drop clients', 'time constraints',
        'threatening to look elsewhere', 'loss of interest', 'change in business structure',
        'irrelevant requests', 'not the right time', 'not happy', 'negative'
    ]
    for keyword in dissatisfied_keywords:
        if keyword in text_low:
            return 'dissatisfied'

    # User explicitly wants 'yes' to satisfied, 'no' to dissatisfied
    # Assuming these could appear as standalone sentiment if not caught by keywords above
    if text_low == 'yes':
        return 'satisfied'
    if text_low == 'no':
        return 'dissatisfied'

    # Explicitly mentioned positive/neutral categories
    if 'satisfied' in text_low or 'protecting people' in text_low or  'later satisfied' in text_low:
        return 'satisfied'
    if 'neutral' in text_low or 'later neutral' in text_low:
        return 'neutral'

    # Default for 'not discussed' and anything else unclassified
    return 'not discussed'

# Apply the cleaning function to the 'crm_contractor_sentiment' column
df['crm_contractor_sentiment_clean'] = emails['crm_contractor_sentiment'].apply(clean_contractor_sentiment)
emails['crm_contractor_sentiment'] = df['crm_contractor_sentiment_clean']
# Display the value counts for the new cleaned column
print(df['crm_contractor_sentiment_clean'].value_counts())

crm_contractor_sentiment_clean
neutral          53576
not discussed    52202
satisfied        11426
dissatisfied      6185
Name: count, dtype: int64


# Handle crm_dts_or_ssip_mentioned , crm_customer_payment_intention , crm_membership_overdue Column Values

In [543]:
allowed_values = ['yes', 'no', 'not discussed']

initial_rows = emails.shape[0]

emails = emails[emails['crm_dts_or_ssip_mentioned'].isin(allowed_values)].copy()

emails = emails[emails['crm_customer_payment_intention'].isin(allowed_values)].copy()

emails = emails[emails['crm_membership_overdue'].isin(allowed_values)].copy()

# Handle crm_membership_level Column Values

In [544]:
def clean_membership_level(text):
    text_low = str(text).strip().lower()

    if 'accredited' in text_low:
        return 'accredited'
    elif 'assist' in text_low:
        return 'assisted'
    elif 'standard' in text_low:
        return 'standard'
    elif 'silver' in text_low:
        return 'silver'
    elif 'gold' in text_low:
        return 'gold'
    elif 'bronze' in text_low:
        return 'bronze'
    elif 'prem' in text_low or 'premier' in text_low:
        return 'premium'
    elif 'member' in text_low or 'members' in text_low:
        return 'member'
    elif 'in progress' in text_low:
        return 'in progress'
    elif 'express' in text_low:
        return 'express'
    elif 'band' in text_low:
        return 'band'
    else:
        return 'not discussed'

df['crm_membership_level_clean'] = emails['crm_membership_level'].apply(clean_membership_level)

emails['crm_membership_level'] = df['crm_membership_level_clean']

print("Value counts for 'crm_membership_level' after cleaning:")
print(emails['crm_membership_level'].value_counts())

Value counts for 'crm_membership_level' after cleaning:
crm_membership_level
in progress      45211
not discussed    41035
accredited       35944
member             862
standard           191
premium             23
silver              19
gold                17
bronze               5
band                 4
express              4
assisted             1
Name: count, dtype: int64


# Handle crm_customer_complained Column Values

In [545]:
def clean_customer_complained(text):
    text_low = str(text).strip().lower()

    # Rule: if it starts with 'not applicable', it is 'yes'
    if text_low.startswith('not applicable'):
        return 'yes'

    # Standard 'yes' rule
    if 'yes' in text_low:
        return 'yes'

    # Standard 'no' rule
    if 'no' in text_low:
        return 'no'

    # Map 'none' to 'not discussed' as per general cleaning strategy
    if 'none' in text_low:
        return 'not discussed'

    # Default for anything else
    return 'not discussed'

# Apply the cleaning function to the 'crm_customer_complained' column
df['crm_customer_complained_clean'] = emails['crm_customer_complained'].apply(clean_customer_complained)

# Overwrite the original column with the cleaned values
emails['crm_customer_complained'] = df['crm_customer_complained_clean']

# Display the value counts for the cleaned column
print(emails['crm_customer_complained'].value_counts())

crm_customer_complained
no     115764
yes      7552
Name: count, dtype: int64


# Handle crm_financial_hardship_mentioned Column Values

In [546]:
def clean_hardship_value(text):
    text_low = str(text).strip().lower()
    if text_low == 'yes':
        return 'yes'
    elif text_low == 'no':
        return 'no'
    else:
        return 'not discussed'

emails['crm_financial_hardship_mentioned'] = emails['crm_financial_hardship_mentioned'].apply(clean_hardship_value)
print(emails['crm_financial_hardship_mentioned'].value_counts())

crm_financial_hardship_mentioned
not discussed    83671
no               33108
yes               6537
Name: count, dtype: int64


# Handle crm_dissatisfaction_with_support Column Values

In [547]:
def clean_dissatisfaction_value(text):
    text_low = str(text).strip().lower()

    # Keywords indicating 'yes' (dissatisfaction)
    yes_keywords = [
         'yes','negative', 'frustration', 'dissatisfied', 'unhappy', 'difficulties',
        'confusion', 'concern', 'awkward interaction', 'excessive information',
        'overcharging', 'delayed', 'unfulfilled promises', 'incorrect', 'misled',
        'price increase', 'lack of value', 'annoyance', 'struggle', 'issue',
        'error', 'disproportionately affect', 'not met', 'not matching',
        'unwanted call', 'abrupt hang-up', 'shut it down', 'unfair', 'unsustainable'
    ]
    for keyword in yes_keywords:
        if keyword in text_low:
            return 'yes'

    # Keywords indicating 'no'
    no_keywords = ['no']
    for keyword in no_keywords:
        if keyword in text_low:
            return 'no'

    # Keywords indicating 'not applicable' or 'not discussed'
    not_applicable_keywords = [
        'not discussed', 'not applicable', 'no email content provided', 'there is no email content to analyze'
    ]
    for keyword in not_applicable_keywords:
        if keyword in text_low:
            return 'not applicable'

    # If none of the above, default to 'not applicable'
    return 'not applicable'

# Apply the cleaning function to the 'crm_dissatisfaction_with_support' column
df['crm_dissatisfaction_with_support_clean'] = emails['crm_dissatisfaction_with_support'].apply(clean_dissatisfaction_value)

# Overwrite the original column with the cleaned values
emails['crm_dissatisfaction_with_support'] = df['crm_dissatisfaction_with_support_clean']

# Display the value counts for the cleaned column
print(emails['crm_dissatisfaction_with_support'].value_counts())

crm_dissatisfaction_with_support
no                112593
yes                10710
not applicable        13
Name: count, dtype: int64


# Handle crm_negative_customer_experience Column Values

In [548]:
def clean_negative_experience_value(text):
    text_low = str(text).strip().lower()

    # Keywords indicating 'yes' (negative experience)
    yes_keywords = [
        'yes', 'concerns about the audit process', 'causing them to lose work',
        'due to the issue with', 'frustration', 'negative customer experience'
    ]
    for keyword in yes_keywords:
        if keyword in text_low:
            return 'yes'

    # Keywords indicating 'no'
    no_keywords = ['no']
    for keyword in no_keywords:
        if keyword in text_low:
            return 'no'

    # Keywords indicating 'not discussed' or 'not applicable'
    not_discussed_keywords = [
        'not discussed', '[yes/no/not discussed]', 'not applicable (no email content provided)',
        'not applicable (there is no email content to analyze)'
    ]
    for keyword in not_discussed_keywords:
        if keyword in text_low:
            return 'not discussed'

    # Default to 'not discussed' for any other unclassified values
    return 'not discussed'

# Apply the cleaning function to the 'crm_negative_customer_experience' column
df['crm_negative_customer_experience_clean'] = emails['crm_negative_customer_experience'].apply(clean_negative_experience_value)

# Overwrite the original column with the cleaned values
emails['crm_negative_customer_experience'] = df['crm_negative_customer_experience_clean']

# Display the value counts for the cleaned column
print(emails['crm_negative_customer_experience'].value_counts())

crm_negative_customer_experience
no     104998
yes     18318
Name: count, dtype: int64


# Handle crm_refund_mentioned Column Value

In [549]:
def clean_value(text):
    text_low = str(text).strip().lower()

    if text_low == 'yes':
        return 'yes'
    elif text_low == 'no':
        return 'no'
    elif text_low in ['not discussed', 'not applicable', '[yes/no]', 'not applicable (no email content provided)', 'not applicable (there is no email content to analyze)', 'not applicable (no conversation provided)']:
        return 'not applicable'
    else:
        # For all other specific values that are not 'yes' or 'no', categorize as 'not applicable'
        # Based on the user's previous request to keep specific values as 'not applicable'
        return 'not applicable'
# Apply the cleaning function to the 'crm_refund_mentioned' column
emails['crm_refund_mentioned'] = emails['crm_refund_mentioned'].apply(clean_value)

# Display the value counts for the cleaned column
print(emails['crm_refund_mentioned'].value_counts())

crm_refund_mentioned
no                110101
not applicable     11521
yes                 1694
Name: count, dtype: int64


# Handle crm_competitors_mentioned Column Values

In [550]:
def clean_competitors_mentioned(text):
    text_low = str(text).strip().lower()

    # Keywords indicating 'yes' (negative experience)
    if 'yes' in text_low:
        return 'yes'

    # Keywords indicating 'no'
    elif 'no' in text_low:
        return 'no'

    # Default to 'not discussed' for any other unclassified values
    else:
        return 'not discussed'

# Apply the cleaning function to the 'crm_competitors_mentioned' column
df['crm_competitors_mentioned_clean'] = emails['crm_competitors_mentioned'].apply(clean_competitors_mentioned)

# Overwrite the original column with the cleaned values
emails['crm_competitors_mentioned'] = df['crm_competitors_mentioned_clean']

# Display the value counts for the cleaned column
print(df['crm_competitors_mentioned'].value_counts())

crm_competitors_mentioned
No                                           79549
Not Discussed                                26957
Yes                                           5724
Yes (Avetta is mentioned as a competitor)        3
Yes (Avetta mentioned as a competitor)           1
Name: count, dtype: int64


# Aggregation Function to Remove Co_Ref Duplicate Values

In [551]:

emails['year'] = pd.to_numeric(emails['year'], errors='coerce')
numeric_cols = ['crm_contractor_sentiment_score', 'crm_agent_chase_count', 'year']
categorical_cols = [col for col in emails.columns if col not in numeric_cols and col != 'Co_Ref']
aggregation_dict = {}
for col in numeric_cols:
    aggregation_dict[col] = 'mean'
for col in categorical_cols:
    aggregation_dict[col] = lambda x: x.mode()[0] if not x.mode().empty else 'not discussed'
initial_rows = emails.shape[0]
emails_aggregated = emails.groupby('Co_Ref').agg(aggregation_dict).reset_index()
final_rows = emails_aggregated.shape[0]
emails = emails_aggregated
print(f"Initial rows: {initial_rows}")
print(f"Final rows after consolidating duplicates based on 'Co_Ref': {final_rows}")
print(f"Number of rows consolidated: {initial_rows - final_rows}")
display(emails.head())

Initial rows: 123316
Final rows after consolidating duplicates based on 'Co_Ref': 37963
Number of rows consolidated: 85353


,Co_Ref,crm_contractor_sentiment_score,crm_agent_chase_count,year,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,...,crm_agent_chased_contractor,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned
0,AA0584,50.000000,1.000000,2025.0,45_out,no,no,yes,no,no,...,no,no,not discussed,0,no,no,no,no,no,not discussed
1,AA0641,16.666667,0.666667,2025.0,14_out,no,not discussed,yes,yes,not discussed,...,no,not discussed,not discussed,0,not discussed,no,no,no,no,not discussed
2,AA0750,50.000000,1.500000,2025.0,45_out,no,no,yes,no,no,...,yes,not discussed,no,0,not discussed,no,no,no,no,no
3,AA0784,0.000000,0.750000,2026.0,14_out,not discussed,not discussed,not discussed,not discussed,not discussed,...,yes,no,no,0,no,no,no,no,no,no
4,AA0794,16.666667,1.666667,2025.0,14_out,no,not discussed,yes,yes,no,...,yes,not discussed,yes,0,not discussed,no,no,no,no,no


# Duplicate Percenatge For Co_Ref After Aggregation

In [552]:
total_rows = emails.shape[0]

duplicate_rows = emails[emails.duplicated(subset=['Co_Ref','Time_to_Renewal','year'], keep=False)].shape[0]

percentage = (duplicate_rows / total_rows) * 100

print("Total rows:", total_rows)
print("Duplicated rows:", duplicate_rows)
print("Percentage of duplicates: {:.2f}%".format(percentage))

Total rows: 37963
Duplicated rows: 0
Percentage of duplicates: 0.00%


# Considering Feature for Churn Analysis

In [553]:

selected_churn_features = [
    'Co_Ref',
    'crm_contractor_sentiment_score',
    'crm_agent_chase_count',
    'Time_to_Renewal',
    'crm_delays_in_accreditation',
    'crm_contractor_engagement',
    'crm_customer_payment_intention',
    'crm_competitors_mentioned',
    'crm_membership_overdue',
    'crm_customer_complained',
    'crm_dissatisfaction_with_support'
]
display(emails[selected_churn_features].head())

,Co_Ref,crm_contractor_sentiment_score,crm_agent_chase_count,Time_to_Renewal,crm_delays_in_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_overdue,crm_customer_complained,crm_dissatisfaction_with_support
0,AA0584,50.000000,1.000000,45_out,no,yes,not discussed,no,not discussed,no,no
1,AA0641,16.666667,0.666667,14_out,yes,yes,not discussed,no,not discussed,no,no
2,AA0750,50.000000,1.500000,45_out,no,yes,not discussed,no,no,no,no
3,AA0784,0.000000,0.750000,14_out,not discussed,not discussed,not discussed,no,no,no,no
4,AA0794,16.666667,1.666667,14_out,yes,yes,no,no,yes,no,no


# Label Encoding

In [554]:
from sklearn.preprocessing import LabelEncoder
churn_features_encoded = emails[selected_churn_features].copy()
categorical_churn_features = [col for col in churn_features_encoded.select_dtypes(include='object').columns if col != 'Co_Ref']
print(f"Categorical features to be label encoded: {list(categorical_churn_features)}")
# Apply Label Encoding to each categorical column
label_encoders = {}
for col in categorical_churn_features:
    le = LabelEncoder()
    churn_features_encoded[col] = le.fit_transform(churn_features_encoded[col])
    label_encoders[col] = le
# Display the first few rows of the encoded features
print("\nDataFrame with label encoded churn features (first 5 rows):")
display(churn_features_encoded.head())
# Optionally, display value counts for one of the encoded columns to show the effect
if 'Time_to_Renewal' in categorical_churn_features:
    print("\nValue counts for 'Time_to_Renewal' after label encoding:")
    print(churn_features_encoded['Time_to_Renewal'].value_counts())

Categorical features to be label encoded: ['Time_to_Renewal', 'crm_delays_in_accreditation', 'crm_contractor_engagement', 'crm_customer_payment_intention', 'crm_competitors_mentioned', 'crm_membership_overdue', 'crm_customer_complained', 'crm_dissatisfaction_with_support']

DataFrame with label encoded churn features (first 5 rows):


,Co_Ref,crm_contractor_sentiment_score,crm_agent_chase_count,Time_to_Renewal,crm_delays_in_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_overdue,crm_customer_complained,crm_dissatisfaction_with_support
0,AA0584,50.000000,1.000000,1,0,1,1,0,1,0,0
1,AA0641,16.666667,0.666667,0,2,1,1,0,1,0,0
2,AA0750,50.000000,1.500000,1,0,1,1,0,0,0,0
3,AA0784,0.000000,0.750000,0,1,0,1,0,0,0,0
4,AA0794,16.666667,1.666667,0,2,1,0,0,2,0,0



Value counts for 'Time_to_Renewal' after label encoding:
Time_to_Renewal
0    27455
1     6692
2     2428
3     1388
Name: count, dtype: int64


# Download Features as the csv file

In [555]:
from google.colab import files

output_filename = 'churn_features_encoded.csv'
churn_features_encoded.to_csv(output_filename, index=False)
files.download(output_filename)
print(f"The encoded churn features have been saved to '{output_filename}' and are ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The encoded churn features have been saved to 'churn_features_encoded.csv' and are ready for download.


# Left Join the Renewal Data and the Feature Data

In [556]:

renewal_calls_df = pd.read_csv('renewal_calls_data.csv')
churn_features_encoded_df = pd.read_csv('churn_features_encoded.csv')
churn_features_encoded_df['Co_Ref'] = churn_features_encoded_df['Co_Ref'].astype(str)
merged_df = pd.merge(renewal_calls_df, churn_features_encoded_df, on='Co_Ref', how='left')
print("Shape of renewal_calls_df:", renewal_calls_df.shape)
print("Shape of churn_features_encoded_df:", churn_features_encoded_df.shape)
print("Shape of merged_df:", merged_df.shape)
print("\nFirst 5 rows of the merged DataFrame:")
display(merged_df.head())

Shape of renewal_calls_df: (35839, 25)
Shape of churn_features_encoded_df: (37963, 11)
Shape of merged_df: (35839, 35)

First 5 rows of the merged DataFrame:


,Co_Ref,Total_Calls_In_Window,Call_Date,desire_to_cancel_clean,Customer_Reaction_Category,Customer_Asked_For_Justification,Membership_Renewal_Decision,Explicit_Switching_Intent,mid_price_log,mid_price_log_flag,...,crm_contractor_sentiment_score,crm_agent_chase_count,Time_to_Renewal,crm_delays_in_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_overdue,crm_customer_complained,crm_dissatisfaction_with_support
0,AA0584,2,2024-05-01,Not Discussed,Not Mentioned,NaN,NaN,NaN,NaN,1,...,50.000000,1.000000,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,AA0641,3,2024-11-11,Not Discussed,Not Mentioned,NaN,NaN,NaN,NaN,1,...,16.666667,0.666667,0.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0
2,AA0784,3,2026-01-16,Not Discussed,Not Mentioned,No,No,No,NaN,1,...,0.000000,0.750000,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,AA0794,7,2025-10-16,Not Discussed,Not Mentioned,No,No,No,NaN,1,...,16.666667,1.666667,0.0,2.0,1.0,0.0,0.0,2.0,0.0,0.0
4,AA0882,1,2025-09-01,Renew,Not Mentioned,No,No,No,NaN,1,...,50.000000,2.000000,1.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0


# Download the Merged Data for Model Building

In [557]:
from google.colab import files

# Save the merged DataFrame to a new CSV file
output_filename_merged = 'merged_churn_data.csv'
merged_df.to_csv(output_filename_merged, index=False)

# Provide a link for download
files.download(output_filename_merged)

print(f"The merged DataFrame has been saved to '{output_filename_merged}' and is ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The merged DataFrame has been saved to 'merged_churn_data.csv' and is ready for download.
